In [ ]:
# Conditional Sequential VAE for 60-month yield-curve scenario generation
# ML engineer + actuary perspective.
#
# Model:  p(Delta_Y_{1:60} | Y_0)
#   * GRU encoder reads the future delta-yield path conditioned on the current curve.
#   * GRU decoder generates the path autoregressively, conditioned on z + Y_0.
#   * Scheduled sampling so the decoder is trained for free-running generation.
#
# Loss:  L_recon  + beta * L_KL  + lam_smooth * L_smooth
#                 + lam_corr  * L_corr  + lam_vol * L_vol  + lam_tail * L_tail
#
# Output:  200 scenarios, each 60 monthly curves, saved as scenarios.csv with
# columns scenario_id, projection_month, projection_date, yield_1m, ..., yield_30y.

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
torch.manual_seed(0); np.random.seed(0)

# ============================================================
# CONFIG
# ============================================================
HORIZON      = 60
LATENT_DIM   = 16
HIDDEN_DIM   = 96
COND_DIM     = 32
NUM_LAYERS   = 1
DROPOUT      = 0.0

EPOCHS       = 80
BATCH_SIZE   = 32
LR           = 5e-4
WEIGHT_DECAY = 1e-5

# Loss weights
BETA_KL          = 0.5     # KL
KL_WARMUP_EPOCHS = 15
LAMBDA_SMOOTH    = 0.05    # temporal + cross-maturity smoothness on recon path
LAMBDA_CORR      = 2.0     # cross-maturity correlation match (free-run vs true)
LAMBDA_VOL       = 2.0     # per-tenor std match  (preserves vol term structure)
LAMBDA_TAIL      = 1.0     # quantile match at 1/5/95/99% (preserves tails)

# Scheduled sampling
TF_START = 1.0
TF_END   = 0.3

# Generation
NUM_SCENARIOS = 200

# Files / split
RAW_DATA_PATH    = "raw_data.csv"
TRAIN_START      = "2005-01-01"
TRAIN_END        = "2023-12-31"
OUTPUT_SCENARIOS = "scenarios.csv"
CHECKPOINT_PATH  = "cvae_checkpoint.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Map FRED column names -> PRD column names
TENOR_RENAME = {
    "Y_DGS1MO": "yield_1m", "Y_DGS3MO": "yield_3m", "Y_DGS6MO": "yield_6m",
    "Y_DGS1":   "yield_1y", "Y_DGS2":   "yield_2y", "Y_DGS3":   "yield_3y",
    "Y_DGS5":   "yield_5y", "Y_DGS7":   "yield_7y", "Y_DGS10":  "yield_10y",
    "Y_DGS20":  "yield_20y","Y_DGS30":  "yield_30y",
}


# ============================================================
# 1. DATA  (drop macro, monthly EOM, filter 2005-2023)
# ============================================================
df_full = pd.read_csv(RAW_DATA_PATH, index_col=0, parse_dates=True).sort_index()

macro_cols = [c for c in ["GDP", "FedRate", "FedFunds", "CPI"] if c in df_full.columns]
if macro_cols:
    print(f"Dropping macro columns: {macro_cols}")
    df_full = df_full.drop(columns=macro_cols)

df_full = df_full.resample("ME").last().dropna()
yield_cols = list(df_full.columns)
TENORS = len(yield_cols)
print(f"Tenors ({TENORS}):", yield_cols)
print("Full monthly range:", df_full.index.min().date(), "->", df_full.index.max().date())

df = df_full.loc[TRAIN_START:TRAIN_END]
print(f"Training window {TRAIN_START} -> {TRAIN_END}: {df.shape}")


# ============================================================
# 2. COMPUTE LEVELS + CHANGES, NORMALISE
# ============================================================
y_arr  = df[yield_cols].values.astype(np.float32)             # levels
dy_arr = y_arr[1:] - y_arr[:-1]                               # monthly changes

# Per-tenor stats (training window only)
mu_y,  std_y  = y_arr.mean(0),  y_arr.std(0)  + 1e-8
mu_dy, std_dy = dy_arr.mean(0), dy_arr.std(0) + 1e-8
print("Empirical Δy std per tenor:", np.round(std_dy, 4))

y_norm  = (y_arr  - mu_y)  / std_y
dy_norm = (dy_arr - mu_dy) / std_dy


# ============================================================
# 3. ROLLING WINDOWS:  (Y_0_norm,  ΔY_{1:60}_norm)
# ============================================================
N = len(y_arr) - HORIZON - 1
y0_list = []
path_list = []
for i in range(N):
    y0_list.append(y_norm[i])              # condition at month i
    path_list.append(dy_norm[i:i+HORIZON]) # 60 future monthly changes
y0_arr   = np.stack(y0_list)               # (N, 11)
path_arr = np.stack(path_list)             # (N, 60, 11)
print("Train windows:", path_arr.shape, " conditions:", y0_arr.shape)


class CondPathDataset(Dataset):
    def __init__(self, y0, paths):
        self.y0    = torch.tensor(y0,    dtype=torch.float32)
        self.paths = torch.tensor(paths, dtype=torch.float32)
    def __len__(self): return len(self.y0)
    def __getitem__(self, i): return self.paths[i], self.y0[i]


train_ds = CondPathDataset(y0_arr, path_arr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


# ============================================================
# 4. MODEL — Conditional Sequential VAE
# ============================================================
class CondSeqVAE(nn.Module):
    def __init__(self, horizon=60, tenors=11, latent_dim=16, hidden_dim=96,
                 cond_dim=32, num_layers=1, dropout=0.0):
        super().__init__()
        self.horizon, self.tenors = horizon, tenors
        self.latent_dim, self.hidden_dim, self.cond_dim, self.num_layers = (
            latent_dim, hidden_dim, cond_dim, num_layers,
        )

        # condition embedding from current curve Y_0
        self.cond_net = nn.Sequential(
            nn.Linear(tenors, cond_dim), nn.LayerNorm(cond_dim), nn.SiLU(),
            nn.Linear(cond_dim, cond_dim), nn.SiLU(),
        )

        # encoder reads the path, then we concat with the cond embedding
        self.encoder_gru = nn.GRU(tenors, hidden_dim,
                                  num_layers=num_layers, batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.encoder_post = nn.Sequential(
            nn.Linear(hidden_dim + cond_dim, hidden_dim),
            nn.LayerNorm(hidden_dim), nn.SiLU(),
        )
        self.mu_head     = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

        # decoder hidden state initialised from z + cond
        self.init_hidden_net = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, hidden_dim * num_layers),
            nn.Tanh(),
        )
        # at each step: prev_dy + cond + z -> hidden -> next dy
        self.decoder_gru = nn.GRU(tenors + cond_dim + latent_dim,
                                  hidden_dim, num_layers=num_layers,
                                  batch_first=True,
                                  dropout=dropout if num_layers > 1 else 0.0)
        self.output_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.SiLU(),
            nn.Linear(hidden_dim, tenors),
        )

    def encode(self, path, y0):
        cond = self.cond_net(y0)
        _, h_n = self.encoder_gru(path)
        h = self.encoder_post(torch.cat([h_n[-1], cond], dim=1))
        return self.mu_head(h), self.logvar_head(h), cond

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def _init_decoder_hidden(self, z, cond):
        h0 = self.init_hidden_net(torch.cat([z, cond], dim=1))
        return h0.view(-1, self.num_layers, self.hidden_dim).transpose(0, 1).contiguous()

    def decode(self, z, y0, teacher_path=None, tf_ratio=0.0):
        B = z.size(0)
        cond = self.cond_net(y0)
        hidden = self._init_decoder_hidden(z, cond)
        prev_y = torch.zeros(B, self.tenors, device=z.device)
        outs = []
        for t in range(self.horizon):
            step = torch.cat([prev_y, cond, z], dim=1).unsqueeze(1)
            out, hidden = self.decoder_gru(step, hidden)
            pred = self.output_head(out.squeeze(1))
            outs.append(pred.unsqueeze(1))
            if teacher_path is not None and tf_ratio > 0:
                mask = (torch.rand(B, device=z.device) < tf_ratio).float().unsqueeze(-1)
                prev_y = mask * teacher_path[:, t, :] + (1 - mask) * pred
            else:
                prev_y = pred
        return torch.cat(outs, dim=1)

    def forward(self, path, y0, tf_ratio=1.0):
        mu, logvar, _ = self.encode(path, y0)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, y0, teacher_path=path, tf_ratio=tf_ratio)
        return recon, mu, logvar


# ============================================================
# 5. LOSSES
# ============================================================
def kl_divergence(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())


def smoothness_loss(path):
    # Penalise sharp jumps both in time and across maturities
    temporal  = F.mse_loss(path[:, 1:, :],  path[:, :-1, :])
    maturity  = F.mse_loss(path[:, :, 1:],  path[:, :, :-1])
    return temporal + maturity


def _corr_matrix(x, eps=1e-8):
    x_c = x - x.mean(dim=0, keepdim=True)
    cov = (x_c.transpose(0, 1) @ x_c) / max(x_c.shape[0] - 1, 1)
    d = torch.sqrt(torch.diag(cov).clamp_min(eps))
    return cov / (d.unsqueeze(0) * d.unsqueeze(1))


def correlation_loss(gen, true):
    g = _corr_matrix(gen.reshape(-1, gen.shape[-1]))
    t = _corr_matrix(true.reshape(-1, true.shape[-1]))
    return F.mse_loss(g, t)


def volatility_loss(gen, true):
    # Match per-tenor std; the empirical std per tenor already encodes the
    # vol term structure (short tenors more volatile than long), so matching
    # it preserves that property.
    return F.mse_loss(gen.std(dim=(0, 1)), true.std(dim=(0, 1)))


def tail_loss(gen, true, qs=(0.01, 0.05, 0.95, 0.99)):
    g = gen.reshape(-1, gen.shape[-1])
    t = true.reshape(-1, true.shape[-1])
    parts = []
    for q in qs:
        parts.append(F.mse_loss(torch.quantile(g, q, dim=0),
                                torch.quantile(t, q, dim=0)))
    return torch.stack(parts).mean()


def get_beta(epoch, max_beta, warmup):
    return max_beta * min(1.0, epoch / warmup)


def get_tf(epoch, total, start, end):
    frac = min(1.0, epoch / max(total - 1, 1))
    return start + (end - start) * frac


# ============================================================
# 6. TRAINING
# ============================================================
model = CondSeqVAE(
    horizon=HORIZON, tenors=TENORS, latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM, cond_dim=COND_DIM,
    num_layers=NUM_LAYERS, dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

hist = {k: [] for k in ["total","rec","kl","smooth","corr","vol","tail"]}

print("\nTraining...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    beta_kl = get_beta(epoch, BETA_KL, KL_WARMUP_EPOCHS)
    tf      = get_tf(epoch - 1, EPOCHS, TF_START, TF_END)

    accum = {k: [] for k in hist}
    for path, y0 in tqdm(train_loader, desc=f"Ep {epoch:03d}/{EPOCHS} (tf={tf:.2f})"):
        path, y0 = path.to(DEVICE), y0.to(DEVICE)

        # ---- Pass 1: scheduled-sampling reconstruction ----
        recon, mu, logvar = model(path, y0, tf_ratio=tf)
        rec   = F.mse_loss(recon, path)
        kl    = kl_divergence(mu, logvar)
        smooth = smoothness_loss(recon)

        # ---- Pass 2: free-running prior sample for stat-matching losses ----
        z_p   = torch.randn(path.size(0), LATENT_DIM, device=DEVICE)
        gen   = model.decode(z_p, y0, teacher_path=None, tf_ratio=0.0)
        corr  = correlation_loss(gen, path)
        vol   = volatility_loss(gen, path)
        tail  = tail_loss(gen, path)

        total = (rec
                 + beta_kl       * kl
                 + LAMBDA_SMOOTH * smooth
                 + LAMBDA_CORR   * corr
                 + LAMBDA_VOL    * vol
                 + LAMBDA_TAIL   * tail)

        optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        accum["total"].append(total.item())
        accum["rec"].append(rec.item());     accum["kl"].append(kl.item())
        accum["smooth"].append(smooth.item()); accum["corr"].append(corr.item())
        accum["vol"].append(vol.item());     accum["tail"].append(tail.item())

    for k, v in accum.items():
        hist[k].append(float(np.mean(v)))

    print(f"  total {hist['total'][-1]:7.4f} | rec {hist['rec'][-1]:6.4f} "
          f"kl {hist['kl'][-1]:6.4f} smooth {hist['smooth'][-1]:6.4f} "
          f"corr {hist['corr'][-1]:6.4f} vol {hist['vol'][-1]:6.4f} "
          f"tail {hist['tail'][-1]:6.4f}  (beta_kl={beta_kl:.3f})")

# ---- Save final checkpoint ----
torch.save({
    "model_state_dict": model.state_dict(),
    "mu_y":  torch.tensor(mu_y),  "std_y":  torch.tensor(std_y),
    "mu_dy": torch.tensor(mu_dy), "std_dy": torch.tensor(std_dy),
    "yield_cols": yield_cols,
    "horizon": HORIZON, "tenors": TENORS, "latent_dim": LATENT_DIM,
    "hidden_dim": HIDDEN_DIM, "cond_dim": COND_DIM,
    "num_layers": NUM_LAYERS, "dropout": DROPOUT,
}, CHECKPOINT_PATH)
print(f"Saved {CHECKPOINT_PATH}")


# ============================================================
# 7. LOSS CURVES
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, key in zip(axes.flat, ["total","rec","kl","smooth","corr","vol","tail"]):
    ax.plot(hist[key])
    ax.set_title(key); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes.flat[-1].axis("off")
plt.tight_layout(); plt.show()


# ============================================================
# 8. GENERATE 200 SCENARIOS  conditional on the latest curve
# ============================================================
model.eval()

# Use latest available curve from the FULL dataset (post-2023 if present)
Y0      = df_full.iloc[-1][yield_cols].values.astype(np.float32)
Y0_date = df_full.index[-1]
print(f"\nGenerating {NUM_SCENARIOS} scenarios conditioned on curve at {Y0_date.date()}")

with torch.no_grad():
    Y0_norm   = (Y0 - mu_y) / std_y
    y0_tensor = torch.tensor(Y0_norm, dtype=torch.float32, device=DEVICE).unsqueeze(0).repeat(NUM_SCENARIOS, 1)
    z         = torch.randn(NUM_SCENARIOS, LATENT_DIM, device=DEVICE)
    dy_norm_g = model.decode(z, y0_tensor, teacher_path=None, tf_ratio=0.0).cpu().numpy()

# Denormalise back to raw Δy and integrate to levels
dy_g     = dy_norm_g * std_dy.reshape(1, 1, -1) + mu_dy.reshape(1, 1, -1)
levels_g = Y0.reshape(1, 1, -1) + np.cumsum(dy_g, axis=1)   # (200, 60, 11)

# Build the scenarios DataFrame in the PRD's column convention
scenario_dates = pd.date_range(
    start=pd.Timestamp(Y0_date) + pd.offsets.MonthEnd(1),
    periods=HORIZON, freq="ME",
)
out_cols = [TENOR_RENAME.get(c, c) for c in yield_cols]

rows = []
for s in range(NUM_SCENARIOS):
    for t in range(HORIZON):
        row = {
            "scenario_id":      s + 1,
            "projection_month": t + 1,
            "projection_date":  scenario_dates[t].date(),
        }
        for i, oc in enumerate(out_cols):
            row[oc] = levels_g[s, t, i]
        rows.append(row)

scenarios_df = pd.DataFrame(rows)
scenarios_df.to_csv(OUTPUT_SCENARIOS, index=False)
print(f"Saved {len(scenarios_df)} rows to {OUTPUT_SCENARIOS}")
print(scenarios_df.head())


# ============================================================
# 9. ACCEPTANCE-CRITERIA DIAGNOSTICS
# ============================================================
# Empirical reference: monthly Δy on the training set
emp_dy   = dy_arr
emp_std  = emp_dy.std(0)
emp_corr = np.corrcoef(emp_dy.T)
emp_q    = {q: np.quantile(emp_dy, q, axis=0) for q in [0.01, 0.05, 0.5, 0.95, 0.99]}

# Generated Δy  (from the 200 scenarios above)
gen_dy_full = np.diff(
    np.concatenate([np.repeat(Y0[None, None, :], NUM_SCENARIOS, axis=0), levels_g], axis=1),
    axis=1,
)
gen_dy_flat = gen_dy_full.reshape(-1, TENORS)
gen_std     = gen_dy_flat.std(0)
gen_corr    = np.corrcoef(gen_dy_flat.T)
gen_q       = {q: np.quantile(gen_dy_flat, q, axis=0) for q in [0.01, 0.05, 0.5, 0.95, 0.99]}

# (a) Volatility term structure
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(out_cols, emp_std, "o-", label="empirical")
ax[0].plot(out_cols, gen_std, "s-", label="generated")
ax[0].set_title("Δy standard deviation by tenor (vol term structure)")
ax[0].tick_params(axis="x", rotation=45); ax[0].grid(alpha=0.3); ax[0].legend()

# (b) Quantile match (1/5/95/99 %)
qs_to_show = [0.01, 0.05, 0.95, 0.99]
for q in qs_to_show:
    ax[1].plot(out_cols, emp_q[q], "o--", alpha=0.6, label=f"emp q{int(q*100)}")
    ax[1].plot(out_cols, gen_q[q], "s-",  alpha=0.9, label=f"gen q{int(q*100)}")
ax[1].set_title("Tail quantiles of Δy")
ax[1].tick_params(axis="x", rotation=45); ax[1].grid(alpha=0.3)
ax[1].legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

# (c) Correlation matrices
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, M, title in zip(ax, [emp_corr, gen_corr],
                       ["Empirical Δy correlation", "Generated Δy correlation"]):
    im = a.imshow(M, vmin=-1, vmax=1, cmap="RdBu_r")
    a.set_title(title)
    a.set_xticks(range(TENORS)); a.set_xticklabels(out_cols, rotation=90)
    a.set_yticks(range(TENORS)); a.set_yticklabels(out_cols)
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()
print("Max abs corr diff:", np.abs(emp_corr - gen_corr).max().round(4),
      " mean:", np.abs(emp_corr - gen_corr).mean().round(4))

# (d) Sample paths — short, mid, long
def idx(name, default):
    return out_cols.index(name) if name in out_cols else default
short_i, mid_i, long_i = idx("yield_1m", 0), idx("yield_10y", TENORS // 2), idx("yield_30y", TENORS - 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, ti, name in zip(axes, [short_i, mid_i, long_i],
                        [out_cols[short_i], out_cols[mid_i], out_cols[long_i]]):
    for s in range(min(40, NUM_SCENARIOS)):
        ax.plot(scenario_dates, levels_g[s, :, ti], alpha=0.35, lw=1)
    ax.plot(scenario_dates, levels_g[:, :, ti].mean(0), "r-", lw=2, label="mean")
    ax.plot([Y0_date], [Y0[ti]], "ko", label="Y_0")
    ax.set_title(f"{name} — 40 sample scenarios")
    ax.set_ylabel("yield (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# (e) Yield curves at month 60 (terminal)  +  start curve
fig, ax = plt.subplots(figsize=(8, 5))
for s in range(min(80, NUM_SCENARIOS)):
    ax.plot(out_cols, levels_g[s, -1, :], "o-", alpha=0.25, lw=1)
ax.plot(out_cols, Y0, "k-", lw=2.5, label="Y_0 (start)")
ax.plot(out_cols, levels_g[:, -1, :].mean(0), "r-", lw=2, label="gen mean @ month 60")
ax.set_title("Yield curves at month 60 (80 scenarios)")
ax.tick_params(axis="x", rotation=45); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# (f) Diversity check: classify each scenario by its end-of-horizon curve shape
slope = levels_g[:, -1, long_i] - levels_g[:, -1, short_i]    # 30y minus 1m
level = levels_g[:, -1, :].mean(axis=1) - Y0.mean()
shapes = pd.DataFrame({
    "scenario_id": np.arange(1, NUM_SCENARIOS + 1),
    "end_short":   levels_g[:, -1, short_i],
    "end_long":    levels_g[:, -1, long_i],
    "end_slope_30y_minus_1m": slope,
    "level_change_vs_start":  level,
})
def regime(row):
    s = row["end_slope_30y_minus_1m"]; l = row["level_change_vs_start"]
    if s < -0.25: return "inversion"
    if s >  1.5 and l >  0.5: return "steepening_high_rates"
    if s >  1.5 and l < -0.5: return "steepening_low_rates"
    if abs(s) < 0.5 and l >  1.0: return "flattening_high"
    if abs(s) < 0.5 and l < -1.0: return "flattening_low"
    if l >  1.0:  return "high_rate_regime"
    if l < -1.0:  return "low_rate_regime"
    return "parallel/normal"
shapes["regime"] = shapes.apply(regime, axis=1)
print("\nScenario regime distribution (end-of-horizon shape buckets):")
print(shapes["regime"].value_counts())

print("\nDone.")
print(f"  Scenarios CSV:  {OUTPUT_SCENARIOS}")
print(f"  Checkpoint:     {CHECKPOINT_PATH}")
